# NOTEBOOK 01 – TẤN LÊN | CHƯƠNG 1 & 6, PHỤ LỤC

**TV1 – Tấn Lên trực tiếp HUẤN LUYỆN MLP cho mục 4.4 Chương 4; vẫn chịu trách nhiệm Chương 1, 6 và FINAL.**

**Đồ án:** Phân tích dữ liệu đơn hàng và dự đoán nguy cơ giao hàng trễ trong thương mại điện tử.

**Vai trò:** Nhóm trưởng, phụ trách mục **1.1–1.4**, **6.1–6.3** và phụ lục. GitHub Issues **#2** và **#7**.

**Hạn bản nháp Chương 1:** 09–10/11/2026. **Chương 6, phụ lục:** 25–27/11/2026. Hoàn tất dự án 30/11/2026.

> Đây là notebook làm việc có hướng dẫn và các cell thực hành; CHƯA có số liệu chạy trên Olist. Không tự tạo kết quả hoặc nhận định thực nghiệm khi thiếu đầu ra từ Ch3–Ch5.

**Tài liệu lý thuyết:** [01 – Tổng quan & dashboard](../docs/theory/01_tan_len_ch1_ch6.md) · [Hợp đồng dữ liệu](../docs/DATA_CONTRACT.md) · [Đề cương](../docs/REPORT_OUTLINE.md).

## A. PHẦN CHƯƠNG 1 – TỔNG QUAN (1.1–1.4)

### 1.1 Bối cảnh và lý do chọn đề tài
**Bài cần trình bày:** mô tả quá trình đặt–duyệt–xử lý–giao hàng trong TMĐT; rủi ro trễ ảnh hưởng trải nghiệm người mua và xử lý của đội vận hành.

**Gợi ý luận điểm:** đơn hàng chỉ được coi là “trễ” sau khi có kết quả; giá trị của dự báo là **sớm hơn**, tại `order_approved_at`. Không tuyên bố dữ liệu Brazil đại diện Việt Nam.

**Đầu ra cho báo cáo:** 1 đoạn bối cảnh có trích dẫn, 1 sơ đồ nghiệp vụ, 1 đoạn lý do chọn đề tài; trích nguồn vào [REFERENCES](../docs/REFERENCES.md).

### 1.2 Phát biểu bài toán và nhãn
- **Dữ liệu đầu vào hợp lệ**: thuộc tính đã tồn tại tại lúc thanh toán được phê duyệt (`t0=order_approved_at`).
- **Biến mục tiêu**: `is_late=1` nếu **ngày giao thực tế > ngày dự kiến**, ngược lại là 0 trên các đơn có nhãn.
- **Đơn vị quan sát**: `order_id` (một đơn = một dòng sau tổng hợp).
- **Đầu ra mô hình**: xác suất giao trễ và danh sách xếp hạng ưu tiên.
- **Giới hạn**: giai đoạn 2016–2018 và tập nhãn được quan sát; dữ liệu thực tế cần kiểm tra trên tập tải về.

**Câu hỏi bảo vệ cần trả lời:** Tại sao dự đoán ở `order_approved_at` và vì sao không thể dùng `order_delivered_customer_date` làm đặc trưng?

### 1.3 Mục tiêu, câu hỏi nghiên cứu
1. Chuẩn hóa dữ liệu Olist theo đơn hàng, tìm yếu tố liên quan tới tỷ lệ giao trễ.
2. Huấn luyện và so sánh **Random Forest** (ML), **MLP** (DL), với **Dummy** tham chiếu.
3. Đánh giá theo Precision, Recall, F1, **Average Precision (AP)**, cảnh báo sai và Top-K.
4. Xây dựng nguyên mẫu **Streamlit** hiển thị KPI lịch sử và đơn cần ưu tiên.

**Giả thuyết nghiên cứu không phải kết luận:** đặc trưng đã có tại lúc xác nhận đơn có thể mang thông tin về nguy cơ trễ; cần kiểm chứng thực nghiệm.

### 1.4 Phạm vi và giới hạn
- Dữ liệu: Olist, các bảng đơn hàng, khách, chi tiết sản phẩm, seller, geolocation (nếu xử lý hợp lệ).
- Đơn có nhãn: trạng thái delivered, đã duyệt, có ngày dự kiến và ngày giao thực tế.
- Không xem đây là dịch vụ logistics đang vận hành năm 2026.
- Tách **EDA lịch sử** và **dự báo sớm** để tránh lạm dụng thông tin sau giao.
- **Phải trình bày giới hạn thiên lệch chọn mẫu:** chỉ các đơn đã giao thành công được gán nhãn trong phương án hiện tại.

## B. PHẦN CHƯƠNG 6 – DASHBOARD, KIỂM THỬ VÀ KẾT LUẬN

### 6.1 Thiết kế hệ thống
**Kiến trúc đề xuất**: Pipeline Ch3 → model đã chọn trên Validation ở Ch4 → test metrics Ch5 → dashboard Ch6.

**Hai trang giao diện:** (1) lịch sử: KPI số đơn, trễ theo khu vực/tháng; (2) scoring: xác suất, phân mức/Top-K, lọc danh sách ưu tiên.

**Yêu cầu:** Dashboard chỉ dùng mô hình huấn luyện đã đóng băng, không train bằng Test. File chính: `app/app.py`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## THỰC NGHIỆM LIÊN CHƯƠNG – HUẤN LUYỆN MLP (TẤN LÊN/TV1, cung cấp cho mục 4.4)

**Không thay đổi chương báo cáo:** phần huấn luyện này được thực hiện bởi Tấn Lên trong notebook 01, nhưng kết quả được Trúc Chi kiểm chứng và trình bày trong mục **4.4 Chương 4**.

**Điều kiện trước khi chạy:** Lam Linh hoàn thành `orders_model_ready.csv`; Trúc Chi đã chốt và chia sẻ `split_manifest.csv`/`split_metadata.json`. Dùng **chính xác split_id, features, cutoff** của Ngọc Lan (RF) và Trúc Chi (Dummy).

**Chỉ được fit MLP trên Train; lựa chọn cấu hình/ngưỡng theo Validation; tuyệt đối không đánh giá hoặc huấn luyện trên Test tại notebook này.**

**Bàn giao:** `models/tan_len_mlp.joblib`, `data/processed/mlp_val_predictions.csv`, `data/processed/mlp_metadata.json`, bảng cấu hình AP Validation, ngưỡng và biểu đồ loss.

In [ ]:
from src.experiment import (get_shared_splits, SAFE_FEATURES, NUMERIC_FEATURES,
                            CATEGORY_FEATURES, choose_threshold_on_validation,
                            precision_recall_at_k)
import json
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score

splits = get_shared_splits(PROCESSED)
train, val = splits["train"], splits["val"]
split_id = splits["metadata"]["split_id"]
X_train, y_train = train[SAFE_FEATURES], train["is_late"].astype(int)
X_val, y_val = val[SAFE_FEATURES], val["is_late"].astype(int)
print("Split chung:", split_id, "| Train:", len(train), "| Validation:", len(val))
print("Đã khoá Test: notebook TV1 không dùng splits['test'] để chọn mô hình.")
display(pd.DataFrame([{"subset": "Train", "N": len(train), "late_rate":y_train.mean()},
                      {"subset": "Validation", "N": len(val), "late_rate":y_val.mean()}]))

### MLP-01. Tiền xử lý train-only và huấn luyện MLP

`StandardScaler` cho biến số; `SimpleImputer` và `OneHotEncoder` cho biến phân loại. Toàn bộ preprocessing được đặt trong `Pipeline` để tránh rò rỉ Validation. Thử một số cấu hình mạng trên Train, dùng **Average Precision của Validation** để chọn. Có thể điều chỉnh cấu hình sau khi được nhóm phê duyệt, không dùng Test để tinh chỉnh.

**Mô tả học thuật cần ghi:** cấu trúc lớp ẩn, activation, regularization `alpha`, early stopping, số epoch/loss và khả năng overfitting.

In [ ]:
def make_mlp(hidden=(64, 32), alpha=0.001):
    preprocessing = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CATEGORY_FEATURES),
    ])
    return Pipeline([
        ("preprocess", preprocessing),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden, alpha=alpha,
            max_iter=180, early_stopping=True, random_state=42,
        )),
    ])

search_space = [
    {"hidden": (32,), "alpha": 0.001},
    {"hidden": (64, 32), "alpha": 0.001},
    {"hidden": (64, 32), "alpha": 0.01},
]
mlp_candidates, fitted_mlp = [], {}
for k, cfg in enumerate(search_space):
    candidate = make_mlp(**cfg)
    candidate.fit(X_train, y_train)  # ONLY TRAIN
    val_prob = candidate.predict_proba(X_val)[:, 1]  # ONLY VALIDATION
    score = float(average_precision_score(y_val, val_prob))
    mlp_candidates.append({"trial":k, "hidden":str(cfg["hidden"]),
                           "alpha":cfg["alpha"], "AP_validation":score,
                           "epochs":candidate.named_steps["mlp"].n_iter_})
    fitted_mlp[k] = candidate
mlp_trial_table = pd.DataFrame(mlp_candidates).sort_values(
    ["AP_validation","trial"], ascending=[False, True])
display(mlp_trial_table)
best_trial = int(mlp_trial_table.iloc[0]["trial"])
mlp = fitted_mlp[best_trial]
mlp_val_prob = mlp.predict_proba(X_val)[:, 1]
print("TV1 chọn trial MLP theo AP trên Validation:", best_trial)

### MLP-02. Precision, Recall, F1, AP/PR-AUC, FP và Top-K – VALIDATION

`is_late=1` là lớp dương. Chọn ngưỡng bằng F1 Validation và báo FP/FN, AP, Precision@K/Recall@K. Đây là **bảng phát triển mô hình**, không phải kết quả Test cuối cùng. Vĩnh Chi chịu trách nhiệm đánh giá độc lập tất cả model trên Test sau khi Trúc Chi đóng băng dự đoán.

In [ ]:
from src.experiment import classification_metrics
mlp_val_metrics = choose_threshold_on_validation(y_val, mlp_val_prob)
mlp_threshold = float(mlp_val_metrics["threshold"])
display(pd.DataFrame([{"model":"mlp","owner":"Tấn Lên","data":"Validation",
                       "split_id":split_id, **mlp_val_metrics}]))
k_values = sorted(set(min(len(val), k) for k in (50,100,200) if k > 0))
mlp_topk = pd.DataFrame([
    precision_recall_at_k(y_val.to_numpy(), mlp_val_prob, k)
    for k in k_values
])
display(mlp_topk)

mlp_loss = mlp.named_steps["mlp"].loss_curve_
if mlp_loss:
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(range(1, len(mlp_loss)+1), mlp_loss)
    ax.set(xlabel="Epoch", ylabel="Training loss", title="MLP loss (Train)")
    ax.grid(alpha=.25)
    plt.show()
print("Kết quả này là Validation, chưa dùng Test.")

### MLP-03. Bàn giao artifact cho Trúc Chi (Chương 4 – mục 4.4)

- `models/tan_len_mlp.joblib` – MLP Pipeline fitted Train;
- `data/processed/mlp_val_predictions.csv` – danh sách `order_id, y_true, p_mlp` theo thứ tự Validation;
- `data/processed/mlp_metadata.json` – người huấn luyện, `split_id`, SHA256 dataset, feature whitelist, best trial, threshold và metrics Validation;
- `reports/chapters/CH04_MODELING.md` – Trúc Chi trình bày học thuật và đối chiếu với mã thực nghiệm.

**Không push binary model/CSV processed lên GitHub**; chuyển nội bộ, chỉ commit notebook/source/docs. Trúc Chi không fit lại MLP.

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
pred_mlp_val = pd.DataFrame({
    "order_id": val["order_id"].to_numpy(),
    "y_true": y_val.to_numpy(),
    "p_mlp": mlp_val_prob,
})
assert pred_mlp_val["order_id"].is_unique
pred_mlp_val.to_csv(PROCESSED / "mlp_val_predictions.csv", index=False)
joblib.dump(mlp, MODEL_DIR / "tan_len_mlp.joblib")
mlp_metadata = {
    "owner":"Tấn Lên", "model":"mlp",
    "split_id":split_id, "dataset_sha256":splits["metadata"]["csv_sha256"],
    "features":list(SAFE_FEATURES), "validation_n":len(val),
    "best_trial":best_trial, "configuration":search_space[best_trial],
    "validation_metrics":mlp_val_metrics,
    "validation_topk":mlp_topk.to_dict(orient="records"),
    "threshold_origin":"Validation only", "random_state":42,
}
(PROCESSED / "mlp_metadata.json").write_text(
    json.dumps(mlp_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Đã lưu MLP artifact + Validation predictions + metadata.")
print("Bàn giao cho Trúc Chi tích hợp Ch4 và Vĩnh Chi đánh giá Test.")

### Checklist MLP của TV1
- [ ] Dùng `split_id` chung từ Trúc Chi; không tự chia lại dữ liệu.
- [ ] Huấn luyện chỉ trên Train, chọn cấu hình/ngưỡng trên Validation.
- [ ] Có bảng AP/Precision/Recall/F1/FP/Top-K Validation, learning curve/loss và nhận xét.
- [ ] Giao `tan_len_mlp.joblib`, `mlp_val_predictions.csv`, `mlp_metadata.json` và minh chứng PR cho Trúc Chi.
- [ ] Mục 4.4 của báo cáo vẫn do Trúc Chi trình bày dựa trên kết quả TV1. Không dùng Test tại đây.

### 6.1.1 Kiểm tra dữ liệu đầu vào từ các chương trước

Cell dưới chỉ **kiểm tra trạng thái bàn giao**, không tạo số liệu giả. Khi thiếu file, phải yêu cầu đúng thành viên phụ trách gửi đầu ra.

In [ ]:
expected = {
    "Ch3 dữ liệu đơn hàng": PROCESSED / "orders_model_ready.csv",
    "Ch4 kết quả dự đoán Test": PROCESSED / "test_predictions.csv",
    "Ch4 mô hình đã chọn": ROOT / "models" / "champion.joblib",
    "Ch5 bảng đánh giá": ROOT / "reports" / "tables" / "model_metrics.csv",
}
handoff = pd.DataFrame([
    {"hạng mục": title, "đường dẫn": str(path.relative_to(ROOT)), "đã có": path.exists()}
    for title, path in expected.items()
])
display(handoff)
if not handoff["đã có"].all():
    print("Chưa thể kết luận / chạy dashboard dự đoán: đang thiếu minh chứng bàn giao.")

### 6.2 Kiểm thử và hướng dẫn chạy
1. Thử chạy `streamlit run app/app.py` từ thư mục gốc.
2. Thử khi thiếu model và khi model đã được load; app phải có thông báo phù hợp.
3. Thử input thiếu cột, sai định dạng và trường không hợp lệ.
4. Kiểm tra cùng một đầu vào, model trả về cùng một xác suất (khi không thay đổi artifact).
5. Kiểm tra biểu đồ EDA chỉ dùng các bản ghi lịch sử, không trộn vào feature dự báo.
6. Lưu screenshot giao diện và tài liệu README; ghi mô hình và thời điểm tính xác suất.

In [ ]:
import importlib.util
for name in ["streamlit", "joblib", "sklearn"]:
    found = importlib.util.find_spec(name) is not None
    print(f"{name}: {'OK' if found else 'CHƯA CÀI'}")
print("Ứng dụng:", ROOT / "app" / "app.py")
print("Lệnh chạy: streamlit run app/app.py")

### 6.3 Kết luận và hướng phát triển
Điền **sau khi nhận** kết quả Ch5:
- Mục tiêu nào đã hoàn thành và bằng chứng?
- Mô hình nào được chọn và vì sao (kết quả trên tập Validation và Test)?
- Với ngân sách giám sát Top-K, số cảnh báo đúng và sai thực tế?
- Hạn chế: 2016–2018, Brazil, thiên lệch delivered-only, chưa có hệ thống realtime.
- Tương lai: thử dữ liệu logistics tại Việt Nam, theo dõi dịch chuyển phân phối và hiệu chuẩn xác suất.

**Không điền con số khi notebook Ch5 chưa sinh ra bảng đánh giá.**

## C. PHỤ LỤC VÀ CHECKLIST BÀN GIAO
- [ ] Báo cáo Ch1 đủ 1.1–1.4, Ch6 đủ 6.1–6.3 và phụ lục.
- [ ] Phát biểu bài toán, nhãn và mốc dự báo khớp Ch3–Ch5.
- [ ] Có hình minh họa sơ đồ hoạt động, dashboard, kết quả có nguồn.
- [ ] Có hướng dẫn chạy dự án và bảng thư viện/phiên bản.
- [ ] Đã kiểm tra demo Streamlit; có phương án ảnh chụp dự phòng.
- [ ] Link PR/commit và sản phẩm bàn giao được cập nhật ở Google Sheets.

**Cách viết báo cáo:** mỗi mục trình bày *mục đích → phương pháp → minh chứng → diễn giải → giới hạn*; không liệt kê code thuần túy.
- [ ] TV1 đã huấn luyện MLP, bàn giao Validation model/metadata cho Trúc Chi; không train trùng ở Ch4.
